In [39]:
# !pip uninstall openai
# !pip install -U openai

In [40]:
import openai
import os 
from dotenv import load_dotenv
import requests
import pandas as pd
import random

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
client = openai.OpenAI(api_key=OPENAI_API_KEY,)

# def create_prompt(product):
    

#     response = client.responses.create(
#         model="gpt-4o",
#         input=f"""
#     Objective: Generate a list of search queries to prompt into tavily, each seach query seperated with commar, to retrieve web-scraped data for the {product}:

#     Product Design Reviews (user/analyst feedback on product design)

#     Product Improvement Suggestions (how to enhance the product)

#     Functional Design Specifications (technical & usability specs)

#     Competitor Information (similar products, market positioning)

#     Constraints:

#     Queries should be optimized for search engines/web scraping.

#     Include both broad and specific keyword variations.

#     Prioritize authoritative sources (e.g., industry blogs, whitepapers, forums, competitor websites).

#     Do not put in website url

#     Put everything in a single sentence, no breaklines
        

#         """
#     )
def create_prompt(product, refinement=False):
    refinement_instruction = "Refine the search terms to improve data quality and ensure longer, more informative content." if refinement else ""
    
    response =  client.responses.create(
        model="gpt-4o",
        input=f"""
Objective: Generate a list of search queries to prompt into Tavily, each search query separated with commas, to retrieve web-scraped data for the {product}:

Product Design Reviews (user/analyst feedback on product design)
Product Improvement Suggestions (how to enhance the product)
Functional Design Specifications (technical & usability specs)
Competitor Information (similar products, market positioning)

Constraints:
- Queries should be optimized for search engines/web scraping.
- Include both broad and specific keyword variations.
- Prioritize authoritative sources (e.g., industry blogs, whitepapers, forums, competitor websites).
- Do not put in website URLs.
- Put everything in a single sentence, no line breaks.
{refinement_instruction}
"""
            
        
    )
    # return response.choices[0].message.content.split(',')
    return response.output_text.split(',')



In [41]:

from dotenv import load_dotenv
import requests
import pandas as pd

load_dotenv()
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

def webscrape(product):
 
    url = "https://api.tavily.com/search"

    payload = {
        "query": product ,
        "topic": "general",
        "search_depth": "advanced",
        "chunks_per_source": 3,
        "max_results": 100,
        "time_range": None,
        "days": 3,
        "include_answer": True,
        "include_raw_content": False,
        "include_images": False,
        "include_image_descriptions": False,
        # "include_domains": ["https://www.reddit.com/", "https://www.pcgamer.com/","https://www.ign.com/news","https://www.reddit.com/r/GamingChairReviews","https://www.techgearlab.com","https://www.tomshardware.com","https://chairdeskexpert.com","https://www.buzzfeed.com" ],
        "include_domains": [],
        "exclude_domains": ["https://www.youtube.com/"]
    }
    headers = {
        "Authorization": "Bearer " + TAVILY_API_KEY,
        "Content-Type": "application/json"
    }

    response = requests.request("POST", url, json=payload, headers=headers)

    response = response.json()
    # print(response)
    answer = response["answer"]
    # print(answer)
    results_dict = {}
    for result in response["results"]:
        results_dict[result["title"]] = {"url":result["url"], 
                                         "content":result["content"]}
    return answer, results_dict






# product = "Secretlab Titan Evo Lite gaming chair review"
# product = "gaming chair positive product "
# product = "Scrape online reviews for top gaming chairs. Extract common complaints on comfort, durability, and ergonomics. Focus on armrests, lumbar support, and materials."

# product ="Compare gaming chairs vs. ergonomic office chairs  on adjustability, breathability, and posture support. Highlight gaps in gaming chair designs."
# product ="Find technical specs for gaming chair materials (examples: PU leather, memory foam density, frame alloys). Include stress-test results or warranty claims."
# product ="Scrape user complaints about gaming chair assembly (tools, instructions, part alignment)"
product ="gaming chair design issues"

webscrape_result = webscrape(product)
print(webscrape_result)

('Gaming chairs often lack proper lumbar support and ergonomic design, leading to discomfort and health issues. Look for chairs with adjustable features and good ventilation. Prioritize ergonomic support for long-term comfort.', {'The Herman Miller Mirra 2 Gaming Chair - Office Logix Shop': {'url': 'https://www.officelogixshop.com/blogs/news/the-herman-miller-mirra-2-gaming-chair-comfort-and-performance-for-serious-gamers?srsltid=AfmBOoreoNZHT9P1g3hw9YLVZFO5Th6i6qlEcqF0UTLkGBKG-zL8KoI5', 'content': "When it comes to gaming chairs, the market is flooded with options that boast flashy designs and gamer-centric aesthetics. However, beneath the surface, many of these chairs lack the essential ergonomic support needed for prolonged gaming sessions. Common complaints include poor lumbar support, limited adjustability, and materials that don't breathe, leading to discomfort and distraction during critical gaming moments. Over time, these issues can even contribute to chronic pain and posture 

In [42]:
webscrape_result

('Gaming chairs often lack proper lumbar support and ergonomic design, leading to discomfort and health issues. Look for chairs with adjustable features and good ventilation. Prioritize ergonomic support for long-term comfort.',
 {'The Herman Miller Mirra 2 Gaming Chair - Office Logix Shop': {'url': 'https://www.officelogixshop.com/blogs/news/the-herman-miller-mirra-2-gaming-chair-comfort-and-performance-for-serious-gamers?srsltid=AfmBOoreoNZHT9P1g3hw9YLVZFO5Th6i6qlEcqF0UTLkGBKG-zL8KoI5',
   'content': "When it comes to gaming chairs, the market is flooded with options that boast flashy designs and gamer-centric aesthetics. However, beneath the surface, many of these chairs lack the essential ergonomic support needed for prolonged gaming sessions. Common complaints include poor lumbar support, limited adjustability, and materials that don't breathe, leading to discomfort and distraction during critical gaming moments. Over time, these issues can even contribute to chronic pain and post

In [43]:
# mehmeh1 = {}
# for x in create_prompt("Secretlab titan evo lite"):

#     _, results = webscrape(x)
#     mehmeh1.update(results)


    

In [44]:
# webscrape_result[1]["Secretlab Titan Evo Lite review: a more affordable gaming chair"]

In [45]:
# prompt = f"""
# Here are 30 randomly selected rows from a dataset:

# {data_str}

# Based on this data, please check if more than half the data is relevant for product design of {product}.


# """

In [ ]:
def evaluate_batch_usefulness(sampled_entries, product):
    sample_texts = [entry["content"][:500] for entry in sampled_entries]  # limit token size
    joined = "\n\n---\n\n".join(sample_texts)

    prompt = f"""
You are a data analsyt who is evaluating whether each entry in a batch of scraped content about "{product}" is useful. 
"Useful" means it contains any of the following: user/analyst product design feedback, improvement suggestions, feature/technical specifications, or competitor comparisons.

Each entry is separated by "---". For each, respond with "yes" or "no" only. Example:

yes  
no  
yes  
...

Here are the entries:
{joined}
"""
    response =client.responses.create(
        model="gpt-4o",
        input=prompt,
    )
    raw_result = response.output_text.lower().strip()
    results = [line.strip() for line in raw_result.splitlines() if line.strip() in {"yes", "no"}]
    return results

def batch_scrape(product, refinement=False):
    prompts = create_prompt(product, refinement=refinement)
    all_results = {}

    for query in prompts:
        _,results = webscrape(query.strip())
        all_results.update(results)


    if len(all_results) < 30:
        return batch_scrape(product, refinement=True)


    sampled_items = random.sample(list(all_results.values()), min(30, len(all_results)))
    eval_results = evaluate_batch_usefulness(sampled_items, product)
    useful_count = eval_results.count("yes")


    if useful_count >= 15:
        return all_results
    else:
        # print("🔁 Batch not useful enough. Retrying with refined prompts.")
        return batch_scrape(product, refinement=True)


In [51]:
product_name = "Secretlab Titan Evo Lite"
final_data = batch_scrape(product_name)

In [52]:
final_data

{'Secretlab Titan EVO Lite Review - Dutchiee.TV': {'url': 'https://www.dutchiee.tv/news/secretlab-titan-evo-lite-review/',
  'content': 'Overall, I have noticed that Titan EVO Lite is highly efficient for both, gaming and official tasks. The lumbar support system ensures that the spine is properly positioned in a manner that will not lead to tiredness or pains especially when one is being confined to a game for several hours. The chair has an inclined feature which provides a certain range of motion to the user to relax and lean back. [...] In summary, the Titan EVO Lite will be a perfect choice for people who want to use Secretlab chairs but do not want to pay a premium for features they do not need. It may not be packed with all the bells and whishes of the premium type of chairs but the EVO Lite gives out quality, built, and back support that will put it into contention with the best in the gaming chair division. [...] Design and Build Quality\n\nThe Secretlab Titan EVO Lite is the 